# 2 - Pipeline de Pré-Processamento Unificado

**Projeto:** Previsão de Churn — Telecomunicações (FIAP Pós-Tech MLOps)  
**Etapa:** 1 · Preparação de Dados  
**Entrada:** `data/interim/telco_typed.parquet`  
**Saídas:** `data/processed/{train,test}.parquet` e `data/processed/{train,test}.npy`

---

## 🎯 Objetivo

Consolidar os notebooks `1_01_data_cleaning`, `1_02_feature_engineering` e `1_03_preprocessing` em um único pipeline reprodutível, documentado e auditável, que transforma os dados brutos tipados no conjunto de treino/teste pronto para consumo pelos modelos.

---

## 📋 Decisões Técnicas (fonte única: `config.py`)

| Decisão | Valor | Justificativa |
|---|---|---|
| `RANDOM_STATE` | `42` | Reprodutibilidade garantida em todas as operações aleatórias |
| Split treino/teste | `80% / 20%` | Padrão da literatura para ~7k amostras |
| Estratificação | `stratify=y` | Preserva taxa de churn (~26.5%) em ambos os conjuntos |
| Imputação `total_charges` | Mediana | 11 registros com `tenure=0`; mediana preserva distribuição melhor que zero |
| Encoding binário | `OrdinalEncoder([No,Yes])` | Garante No=0, Yes=1 independente da ordem dos dados |
| Encoding nominal (3+ cats) | `OneHotEncoder(drop='first')` | Evita dummy trap; `handle_unknown='ignore'` seguro em produção |
| Normalização numérica | `log1p` + `StandardScaler` | `total_charges` tem skewness=0.962; StandardScaler é essencial para estabilidade de gradiente na MLP |
| Features criadas | 6 novas | Sinal adicional confirmado por Cramer's V e correlação com target |

---

## 🗺️ Roadmap do Notebook

```
[1] Setup & Imports
[2] Carregamento e Snapshot
[3] DATA CLEANING
    [3.1] Padronização de nomes (snake_case)
    [3.2] Validação de shape e schema
    [3.3] Remoção de colunas ID e leakage
    [3.4] Imputação de nulos (total_charges)
    [3.5] Normalização semântica ("No internet service" → "No")
    [3.6] Correção de inconsistências lógicas
    [3.7] Validação de ranges numéricos
    [3.8] Remoção de duplicatas
    [3.9] Resumo da limpeza
[4] FEATURE ENGINEERING
    [4.1] num_services
    [4.2] charges_per_month
    [4.3] is_month_to_month
    [4.4] tenure_group
    [4.5] has_security_support
    [4.6] is_fiber_optic
    [4.7] Validação das features
[5] PREPROCESSING & ENCODING
    [5.1] Separação X / y
    [5.2] Definição dos grupos de transformação
    [5.3] Validação de cobertura
    [5.4] Construção do ColumnTransformer
    [5.5] Train/Test Split estratificado
    [5.6] Fit no treino → Transform em ambos
    [5.7] Validações pós-transformação
[6] PERSISTÊNCIA
    [6.1] Parquet (train + test)
    [6.2] NumPy .npy (train + test)
    [6.3] Serialização do preprocessor.pkl
    [6.4] Resumo final
```


---

## [1] Setup & Imports

Todas as dependências são importadas aqui, centralizadas na primeira célula.  
Constantes de configuração vêm exclusivamente de `churn_telecom.config` —
nenhum valor mágico espalhado no notebook.


In [ ]:
# ── stdlib ─────────────────────────────────────────────────────────────────
import random

# ── terceiros ────────────────────────────────────────────────────────────────
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

In [ ]:
# ── módulo interno ───────────────────────────────────────────────────────────
from config import (
    DATA_INTERIM,
    DATA_PROCESSED,
    MODELS_DIR,
    RANDOM_STATE,
    TARGET,
    COLS_ID,
    COLS_POS,
    COLS_NUM,
    COLS_CAT,
    LABEL_COL,
    DATA_ROWS,
    get_logger,
    to_snake_case,
)

In [ ]:
# ── seed global ──────────────────────────────────────────────────────────────
# Seeds fixados conforme boas práticas de reprodutibilidade do projeto.
# RANDOM_STATE vem do config.py — nunca hardcodar aqui.
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

logger = get_logger("2_vab_preprocessing")
logger.info("Iniciando preprocessamento de dados | 2_vab_preprocessing")

07:34:20 | INFO | Iniciando preprocessamento de dados | 2_vab_preprocessing


---

## [2] Carregamento e Snapshot

Carregamos o dataset tipado gerado pelo notebook `0_01` (tipagem de colunas).  
Um **snapshot** é registrado antes de qualquer transformação para auditoria
do delta ao final de cada etapa — boa prática de rastreabilidade.


In [ ]:
df_raw = pd.read_parquet(DATA_INTERIM / "telco_typed.parquet")
logger.info("Dataset carregado | shape=%s", df_raw.shape)
df_raw.head(3)

07:34:20 | INFO | Dataset carregado | shape=(7043, 20)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1


In [ ]:
# Cópia de trabalho — NUNCA modificar o raw original
df = df_raw.copy()

# Snapshot global antes de qualquer transformação
_shape_raw = df.shape
_nulos_raw = df.isnull().sum().sum()

logger.info(
    "SNAPSHOT INICIAL | linhas=%d | colunas=%d | nulos_totais=%d",
    _shape_raw[0],
    _shape_raw[1],
    _nulos_raw,
)

07:34:20 | INFO | SNAPSHOT INICIAL | linhas=7043 | colunas=20 | nulos_totais=11


---

## [3] Data Cleaning

Etapa de limpeza estrutural do dataset. O objetivo é garantir **qualidade,
consistência e ausência de leakage** antes de qualquer transformação estatística.

As operações são sequenciais e ordenadas por dependência:
padronização → remoção de colunas → imputação → normalização semântica → validações.


### [3.1] Padronização de nomes de colunas para `snake_case`

O dataset IBM Telco usa espaços e CamelCase nos nomes de colunas
(ex: `TotalCharges`, `Monthly Charges`). Padronizamos para `snake_case`
para compatibilidade com Python, pandas e o `config.py`.

A função `to_snake_case` é a **única fonte de verdade** para essa conversão —
importada do `config.py`, usada consistentemente em todos os notebooks.


In [ ]:
snapshot_antes_clean = df.shape
nulos_antes_clean = df.isnull().sum().sum()

# Padronização via to_snake_case do config.py
df.columns = [to_snake_case(col) for col in df.columns]
logger.info("Colunas padronizadas para snake_case | total=%d", len(df.columns))
logger.info("Colunas: %s", list(df.columns))

07:34:20 | INFO | Colunas padronizadas para snake_case | total=20
07:34:20 | INFO | Colunas: ['gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_value']


### [3.2] Validação de shape e schema

Validamos que o dataset tem:
- O número esperado de linhas (`DATA_ROWS = 7043`, definido no `config.py`)
- Todas as colunas esperadas pelo schema do projeto

Uma tolerância de ±5 linhas é aceita para linhas — pequenas variações podem
ocorrer por duplicatas ou registros corrompidos detectados na tipagem.


In [ ]:
# ── Validação de shape ───────────────────────────────────────────────────────
delta_linhas = abs(df.shape[0] - DATA_ROWS)
if delta_linhas <= 5:
    logger.info(
        "Shape OK | linhas=%d (esperado=%d | delta=%d)",
        df.shape[0],
        DATA_ROWS,
        delta_linhas,
    )
else:
    logger.warning(
        "Shape inesperado | linhas=%d | esperado=%d | delta=%d",
        df.shape[0],
        DATA_ROWS,
        delta_linhas,
    )

# ── Validação de schema ───────────────────────────────────────────────────────
expected_cols = {
    to_snake_case(c)
    for c in COLS_ID + COLS_NUM + COLS_CAT + COLS_POS + [TARGET, LABEL_COL]
}
current_cols = set(df.columns)

missing_cols = expected_cols - current_cols
extra_cols = current_cols - expected_cols

if missing_cols:
    logger.warning("Colunas ausentes no df: %s", sorted(missing_cols))
if extra_cols:
    logger.info("Colunas extras (serão avaliadas): %s", sorted(extra_cols))
if not missing_cols:
    logger.info("Schema validado — todas as colunas esperadas presentes.")

07:34:20 | INFO | Shape OK | linhas=7043 (esperado=7043 | delta=0)
07:34:20 | WARNING | Colunas ausentes no df: ['churn_label', 'churn_reason', 'churn_score', 'city', 'cltv', 'count', 'country', 'customerid', 'lat_long', 'latitude', 'longitude', 'state', 'zip_code']


### [3.3] Remoção de colunas sem poder preditivo e com leakage

Duas categorias de colunas são removidas por razões distintas:

**Colunas ID/Geo (`COLS_ID`):** Identificadores únicos de cliente e
informações geográficas que não têm poder preditivo sobre o churn.
Mantê-las causaria *overfitting* em modelos que memorizariam IDs.

**Colunas pós-evento (`COLS_POS`):** Variáveis geradas *após* o evento
de churn (ex: motivo do cancelamento, data da saída). Incluí-las seria
**data leakage** — o modelo aprenderia a detectar o churn com informação
que só existe depois que o churn já ocorreu, tornando a avaliação inválida.

**`LABEL_COL`:** Versão string do target (`"Yes"/"No"`) — redundante
com a coluna numérica `churn_value` (0/1).


In [ ]:
# ── Remoção de colunas ID/Geo ────────────────────────────────────────────────
cols_id_snake = [to_snake_case(c) for c in COLS_ID]
cols_id_present = [c for c in cols_id_snake if c in df.columns]
df.drop(columns=cols_id_present, inplace=True)
logger.info(
    "ID/Geo removidas (%d): %s",
    len(cols_id_present),
    cols_id_present,
)

# ── Remoção de colunas com leakage pós-evento ────────────────────────────────
cols_pos_snake = [to_snake_case(c) for c in COLS_POS]
cols_pos_present = [c for c in cols_pos_snake if c in df.columns]
df.drop(columns=cols_pos_present, inplace=True)
logger.warning(
    "Leakage removido (%d colunas pós-evento): %s",
    len(cols_pos_present),
    cols_pos_present,
)

# ── Remoção de label_col (string redundante do target) ───────────────────────
label_snake = to_snake_case(LABEL_COL)
if label_snake in df.columns:
    df.drop(columns=[label_snake], inplace=True)
    logger.info(
        "label_col removida: '%s' (redundante com target numérico)", label_snake
    )

07:34:20 | INFO | ID/Geo removidas (0): []
07:34:20 | WARNING | Leakage removido (0 colunas pós-evento): []


### [3.4] Garantia e tipagem do target

O target `churn_value` deve ser um inteiro binário (0 = não saiu, 1 = saiu).
Validamos sua presença e forçamos o dtype `int` para compatibilidade com
todos os modelos scikit-learn e PyTorch.

Em caso de ausência da coluna exata, uma busca por colunas com "churn" no
nome garante robustez contra variações de nomenclatura no dataset de entrada.


In [ ]:
TARGET_COL = to_snake_case(TARGET)

# Busca com fallback — robusto a variações de nomenclatura
if TARGET_COL not in df.columns:
    candidates = [c for c in df.columns if "churn" in c.lower()]
    if candidates:
        df.rename(columns={candidates[0]: TARGET_COL}, inplace=True)
        logger.warning("TARGET renomeado: '%s' → '%s'", candidates[0], TARGET_COL)
    else:
        logger.error(
            "TARGET '%s' não encontrado — pipeline não pode continuar.", TARGET_COL
        )
        raise ValueError(f"Coluna target '{TARGET_COL}' ausente no dataset.")

df[TARGET_COL] = df[TARGET_COL].astype(int)
churn_rate_global = df[TARGET_COL].mean() * 100
logger.info(
    "TARGET '%s' validado | dtype=int | churn_rate=%.2f%% | n_churners=%d / %d",
    TARGET_COL,
    churn_rate_global,
    df[TARGET_COL].sum(),
    len(df),
)

07:34:20 | INFO | TARGET 'churn_value' validado | dtype=int | churn_rate=26.54% | n_churners=1869 / 7043


### [3.5] Imputação de valores nulos em `total_charges`

**Diagnóstico:** 11 registros apresentam `total_charges = NaN`.
Todos correspondem a clientes com `tenure_months = 0` (recém-chegados
que ainda não receberam nenhuma cobrança acumulada).

**Decisão: imputação pela mediana.**
- A mediana é mais robusta a outliers do que a média
- Zero seria factualmente incorreto (haverá cobranças futuras)
- A mediana preserva a distribuição original da variável

Esta é a **única imputação** do dataset — todas as demais colunas estão completas.


In [ ]:
total_charges_col = "total_charges"
tenure_col = "tenure_months"

if total_charges_col in df.columns:
    n_nulos_tc = df[total_charges_col].isna().sum()

    # Verificação da hipótese: todos os nulos têm tenure=0?
    if n_nulos_tc > 0 and tenure_col in df.columns:
        tenure_dos_nulos = df.loc[df[total_charges_col].isna(), tenure_col].unique()
        hipotese_ok = bool((tenure_dos_nulos == 0).all())
        logger.info(
            "Hipótese tenure=0 para nulos: %s | tenure dos nulos=%s",
            hipotese_ok,
            tenure_dos_nulos,
        )

    mediana_tc = df[total_charges_col].median()
    df[total_charges_col] = df[total_charges_col].fillna(mediana_tc)

    logger.info(
        "%s | %d nulos imputados com mediana=%.2f | nulos_restantes=%d",
        total_charges_col,
        n_nulos_tc,
        mediana_tc,
        df[total_charges_col].isna().sum(),
    )
else:
    logger.warning("'%s' não encontrada — imputação ignorada.", total_charges_col)

07:34:20 | INFO | Hipótese tenure=0 para nulos: True | tenure dos nulos=[0]
07:34:20 | INFO | total_charges | 11 nulos imputados com mediana=1397.47 | nulos_restantes=0


### [3.6] Normalização semântica — `"No internet service"` → `"No"`

Seis colunas de serviços de internet (segurança, backup, suporte, streaming etc.)
contêm três valores possíveis: `"Yes"`, `"No"` e `"No internet service"`.

**Problema:** `"No"` e `"No internet service"` têm **o mesmo significado preditivo**
— o cliente não possui aquele serviço. Manter os dois valores aumenta a
cardinalidade desnecessariamente, criando mais colunas no OHE e potencial
confusão para o modelo.

**Decisão:** Substituir `"No internet service"` por `"No"` em todas as colunas afetadas,
reduzindo de 3 para 2 categorias por coluna.


In [ ]:
NO_SERVICE_COLS = [
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

total_substituicoes = 0
for col in NO_SERVICE_COLS:
    if col not in df.columns:
        continue
    n_afetados = (df[col] == "No internet service").sum()
    if n_afetados > 0:
        df[col] = df[col].replace({"No internet service": "No"})
        total_substituicoes += n_afetados
        logger.info(
            "%s | 'No internet service' → 'No' | n=%d registros",
            col,
            n_afetados,
        )

logger.info(
    "Normalização semântica concluída | total_substituições=%d em %d colunas",
    total_substituicoes,
    len(NO_SERVICE_COLS),
)

07:34:20 | INFO | online_security | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | online_backup | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | device_protection | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | tech_support | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | streaming_tv | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | streaming_movies | 'No internet service' → 'No' | n=1526 registros
07:34:20 | INFO | Normalização semântica concluída | total_substituições=9156 em 6 colunas


### [3.7] Correção de inconsistências lógicas

**Regra de negócio:** Um cliente sem serviço de internet (`internet_service = "No"`)
não pode ter serviços de internet ativos (segurança online, backup, streaming etc.).

Estas inconsistências podem surgir de erros de entrada de dados no CRM da operadora.
Mantê-las criaria padrões contraditórios no dataset que confundiriam o modelo.

**Correção:** Forçar `"No"` em todas as colunas de serviço de internet
para clientes com `internet_service = "No"`.


In [ ]:
internet_col = "internet_service"

if internet_col in df.columns:
    mask_sem_internet = df[internet_col] == "No"

    for col in NO_SERVICE_COLS:
        if col not in df.columns:
            continue
        n_inconsistentes = (mask_sem_internet & (df[col] != "No")).sum()
        if n_inconsistentes > 0:
            df.loc[mask_sem_internet, col] = "No"
            logger.warning(
                "%s | %d inconsistências corrigidas (sem internet mas serviço ativo)",
                col,
                n_inconsistentes,
            )

    logger.info("Correção de inconsistências lógicas concluída.")

07:34:20 | INFO | Correção de inconsistências lógicas concluída.


### [3.8] Validação de ranges numéricos

Verificamos se as colunas numéricas estão dentro dos intervalos esperados
para o negócio de telecomunicações. Valores fora do range indicam
erros de entrada ou corrupção de dados.

| Coluna | Range esperado | Justificativa |
|---|---|---|
| `tenure_months` | [0, 72] | Contratos de até 6 anos |
| `monthly_charges` | [0, 200] | Planos premium no máximo ~US$120; folga para crescimento |
| `total_charges` | [0, 10.000] | 72 meses × $120 = $8.640; folga incluída |


In [ ]:
RANGE_RULES = {
    "tenure_months": (0, 72),
    "monthly_charges": (0, 200),
    "total_charges": (0, 10_000),
}

for col, (min_v, max_v) in RANGE_RULES.items():
    if col not in df.columns:
        continue
    n_invalidos = ((df[col] < min_v) | (df[col] > max_v)).sum()
    if n_invalidos > 0:
        logger.warning(
            "%s | %d valores fora do range [%.0f, %.0f]",
            col,
            n_invalidos,
            min_v,
            max_v,
        )
    else:
        logger.info("%s | range OK [%.0f, %.0f]", col, min_v, max_v)

07:34:20 | INFO | tenure_months | range OK [0, 72]
07:34:20 | INFO | monthly_charges | range OK [0, 200]
07:34:20 | INFO | total_charges | range OK [0, 10000]


### [3.9] Remoção de duplicatas

Verificamos registros completamente duplicados. No Telco IBM Dataset
não são esperadas duplicatas, mas a verificação é obrigatória como
parte do pipeline de qualidade de dados.


In [ ]:
n_dup = df.duplicated().sum()
if n_dup > 0:
    df.drop_duplicates(inplace=True)
    logger.warning("Duplicatas removidas: %d", n_dup)
else:
    logger.info("Sem duplicatas encontradas.")

07:34:20 | WARNING | Duplicatas removidas: 22


### [3.10] Resumo da limpeza

Consolidação do delta de linhas, colunas e nulos entre o estado inicial e o final.


In [ ]:
shape_depois_clean = df.shape
nulos_depois_clean = df.isnull().sum().sum()

logger.info(
    "RESUMO LIMPEZA | "
    "linhas: %d→%d (delta=%d) | "
    "colunas: %d→%d (removidas=%d) | "
    "nulos: %d→%d",
    snapshot_antes_clean[0],
    shape_depois_clean[0],
    snapshot_antes_clean[0] - shape_depois_clean[0],
    snapshot_antes_clean[1],
    shape_depois_clean[1],
    snapshot_antes_clean[1] - shape_depois_clean[1],
    nulos_antes_clean,
    nulos_depois_clean,
)

# Exibe amostra do dataset limpo
df.head(3)

07:34:20 | INFO | RESUMO LIMPEZA | linhas: 7043→7021 (delta=22) | colunas: 20→20 (removidas=0) | nulos: 11→0


,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn_value
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1


---

## [4] Feature Engineering

Criação de 6 novas features a partir das colunas existentes.
Cada feature foi validada na EDA bivariada e possui justificativa baseada
em correlação com o target ou literatura de churn em telecomunicações.

> **Princípio de separação de responsabilidades:** O encoding (OrdinalEncoder, OHE)
> é responsabilidade exclusiva do `[5] Preprocessing`. Aqui as colunas originais
> permanecem como strings — só criamos novas features.


In [ ]:
# Constantes de colunas — snake_case alinhado com 1.01
TENURE_COL = "tenure_months"
MONTHLY_COL = "monthly_charges"
TOTAL_COL = "total_charges"
CONTRACT_COL = "contract"
INTERNET_COL = "internet_service"
SECURITY_COL = "online_security"
SUPPORT_COL = "tech_support"

SERVICES_COLS = [
    "phone_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

# Validação de pré-condições: todas as colunas necessárias existem
required_for_fe = SERVICES_COLS + [
    TENURE_COL,
    MONTHLY_COL,
    TOTAL_COL,
    CONTRACT_COL,
    INTERNET_COL,
    TARGET_COL,
]
missing_fe = [c for c in required_for_fe if c not in df.columns]
assert not missing_fe, f"Colunas ausentes para feature engineering: {missing_fe}"

shape_antes_fe = df.shape
logger.info(
    "SNAPSHOT ANTES FE | shape=%s | colunas=%s",
    shape_antes_fe,
    list(df.columns),
)

07:34:20 | INFO | SNAPSHOT ANTES FE | shape=(7021, 20) | colunas=['gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_value']


### [4.1] `num_services` — Contagem de serviços ativos

**Hipótese:** Clientes com mais serviços contratados têm maior **custo de saída**
(switching cost). Para cancelar, precisam buscar múltiplos substitutos,
o que funciona como âncora de retenção.

**Método:** Conta quantas das 7 colunas de serviço têm valor `"Yes"`.
O range esperado é [0, 7].

**Evidência empírica (EDA):** Taxa de churn decresce monotonicamente com
`num_services`: de ~35% com 0 serviços para ~8% com 7 serviços.


In [ ]:
cols_services_present = [c for c in SERVICES_COLS if c in df.columns]
df["num_services"] = (df[cols_services_present] == "Yes").sum(axis=1).astype(int)

churn_por_svc = df.groupby("num_services")[TARGET_COL].mean().mul(100).round(1)
logger.info(
    "num_services criado | min=%d | max=%d | média=%.2f | range_completo=%s",
    df["num_services"].min(),
    df["num_services"].max(),
    df["num_services"].mean(),
    list(churn_por_svc.index),
)
for n_svc, churn_rate in churn_por_svc.items():
    logger.info(
        "  num_services=%d | n=%d | churn=%.1f%%",
        n_svc,
        (df["num_services"] == n_svc).sum(),
        churn_rate,
    )

07:34:20 | INFO | num_services criado | min=0 | max=7 | média=2.95 | range_completo=[0, 1, 2, 3, 4, 5, 6, 7]
07:34:20 | INFO |   num_services=0 | n=80 | churn=43.8%
07:34:20 | INFO |   num_services=1 | n=2231 | churn=21.3%
07:34:20 | INFO |   num_services=2 | n=996 | churn=43.5%
07:34:20 | INFO |   num_services=3 | n=1041 | churn=34.7%
07:34:20 | INFO |   num_services=4 | n=1062 | churn=27.2%
07:34:20 | INFO |   num_services=5 | n=827 | churn=22.0%
07:34:20 | INFO |   num_services=6 | n=525 | churn=12.6%
07:34:20 | INFO |   num_services=7 | n=259 | churn=5.8%


### [4.2] `charges_per_month` — Custo relativo ao tempo de casa

**Hipótese:** Um cliente que paga muito mas está há pouco tempo na operadora
tem maior percepção de "custo-benefício ruim" — perfil de alto risco de churn.

**Fórmula:** `monthly_charges / (tenure_months + 1)`
O `+1` evita divisão por zero para clientes com `tenure = 0`.

**Interpretação:** Clientes novos com planos caros terão valores altos;
clientes antigos com planos baratos terão valores baixos.

**Evidência empírica (EDA):** Correlação Pearson de ~0.40 com o target.


In [ ]:
df["charges_per_month"] = (df[MONTHLY_COL] / (df[TENURE_COL] + 1)).round(4)

logger.info(
    "charges_per_month criado | min=%.2f | max=%.2f | média=%.2f | mediana=%.2f | "
    "corr_target=%.4f",
    df["charges_per_month"].min(),
    df["charges_per_month"].max(),
    df["charges_per_month"].mean(),
    df["charges_per_month"].median(),
    df["charges_per_month"].corr(df[TARGET_COL]),
)

07:34:20 | INFO | charges_per_month criado | min=0.26 | max=80.85 | média=5.73 | mediana=2.07 | corr_target=0.4094


### [4.3] `is_month_to_month` — Flag de contrato mensal

**Hipótese:** O tipo de contrato é o maior preditor categórico do churn.
Clientes month-to-month podem cancelar a qualquer momento sem penalidade.

**Evidência empírica (EDA):**
- Cramer's V = 0.41 (associação forte com o target)
- Churn rate month-to-month: **42.71%** vs. ~12% em contratos anuais

**Nota:** A coluna original `contract` é mantida para o OHE no passo [5].
Esta feature binária facilita a interpretação e reduz dimensionalidade
para modelos lineares.


In [ ]:
df["is_month_to_month"] = (df[CONTRACT_COL] == "Month-to-month").astype(int)

n_mtm = df["is_month_to_month"].sum()
n_outros = len(df) - n_mtm
churn_mtm = df.loc[df["is_month_to_month"] == 1, TARGET_COL].mean() * 100
churn_outros = df.loc[df["is_month_to_month"] == 0, TARGET_COL].mean() * 100

logger.info(
    "is_month_to_month | mtm=%d (%.1f%%) churn=%.1f%% | outros=%d (%.1f%%) churn=%.1f%%",
    n_mtm,
    n_mtm / len(df) * 100,
    churn_mtm,
    n_outros,
    n_outros / len(df) * 100,
    churn_outros,
)

07:34:20 | INFO | is_month_to_month | mtm=3853 (54.9%) churn=42.6% | outros=3168 (45.1%) churn=6.8%


### [4.4] `tenure_group` — Segmentação do tempo de casa

**Hipótese:** A relação entre `tenure` e churn é **não-linear** — a curva de
risco cai exponencialmente nos primeiros meses e se estabiliza depois.
Buckets discretos capturam essa não-linearidade que escaparia de modelos lineares.

**Segmentação baseada na EDA:**
- `"novo"` (≤ 12 meses): zona crítica de risco — mediana de churn em ~10 meses
- `"medio"` (13–48 meses): período de estabilização do relacionamento
- `"longo"` (> 48 meses): clientes fidelizados, risco mínimo

**Evidência empírica (EDA):** Cohen's d = −0.89 entre churners e não-churners,
concentrado nos primeiros 12 meses.

**Nota:** Tipada como string → OHE aplicado no passo [5].


In [ ]:
df["tenure_group"] = pd.cut(
    df[TENURE_COL],
    bins=[0, 12, 48, float("inf")],
    labels=["novo", "medio", "longo"],
    right=True,
    include_lowest=True,
).astype(str)

logger.info("tenure_group criado | distribuição e churn por bucket:")
for grupo in ["novo", "medio", "longo"]:
    mask = df["tenure_group"] == grupo
    count = mask.sum()
    cr = df.loc[mask, TARGET_COL].mean() * 100
    logger.info(
        "  tenure_group='%s' | n=%d (%.1f%%) | churn=%.1f%%",
        grupo,
        count,
        count / len(df) * 100,
        cr,
    )

07:34:20 | INFO | tenure_group criado | distribuição e churn por bucket:
07:34:20 | INFO |   tenure_group='novo' | n=2164 (30.8%) | churn=47.4%
07:34:20 | INFO |   tenure_group='medio' | n=2618 (37.3%) | churn=23.6%
07:34:20 | INFO |   tenure_group='longo' | n=2239 (31.9%) | churn=9.5%


### [4.5] `has_security_support` — Flag de proteção consolidada

**Hipótese:** Clientes sem segurança online E sem suporte técnico têm menor
percepção de valor nos serviços da operadora, aumentando o risco de churn.

**Consolidação de duas features correlacionadas:**
- `online_security`: Cramer's V ≈ 0.34
- `tech_support`: Cramer's V ≈ 0.33

Ambas correlacionadas entre si — consolidar em uma feature reduz
dimensionalidade sem perda significativa de sinal.

**Valor da feature:** 1 se tem `online_security = "Yes"` OU `tech_support = "Yes"`.


In [ ]:
df["has_security_support"] = (
    (df[SECURITY_COL] == "Yes") | (df[SUPPORT_COL] == "Yes")
).astype(int)

n_has = df["has_security_support"].sum()
n_sem = len(df) - n_has
cr_has = df.loc[df["has_security_support"] == 1, TARGET_COL].mean() * 100
cr_sem = df.loc[df["has_security_support"] == 0, TARGET_COL].mean() * 100

logger.info(
    "has_security_support | tem=%d (%.1f%%) churn=%.1f%% | "
    "não tem=%d (%.1f%%) churn=%.1f%%",
    n_has,
    n_has / len(df) * 100,
    cr_has,
    n_sem,
    n_sem / len(df) * 100,
    cr_sem,
)

07:34:20 | INFO | has_security_support | tem=2964 (42.2%) churn=17.1% | não tem=4057 (57.8%) churn=33.3%


### [4.6] `is_fiber_optic` — Flag de internet fibra óptica

**Hipótese:** Paradoxalmente, clientes com fibra óptica têm a **maior taxa de churn**
por categoria de internet. Possíveis razões: preço mais alto, maiores expectativas
de qualidade ou maior propensão a pesquisar concorrentes.

**Evidência empírica (EDA):**
- Churn rate fibra: **41.89%** vs. ~19% (DSL) e ~7% (sem internet)
- Cramer's V = 0.32 com o target

Isolar em uma feature binária captura o sinal mais forte de `internet_service`
sem perder a variação OHE da coluna original.


In [ ]:
df["is_fiber_optic"] = (df[INTERNET_COL] == "Fiber optic").astype(int)

n_fiber = df["is_fiber_optic"].sum()
n_outros = len(df) - n_fiber
cr_fiber = df.loc[df["is_fiber_optic"] == 1, TARGET_COL].mean() * 100
cr_rest = df.loc[df["is_fiber_optic"] == 0, TARGET_COL].mean() * 100

logger.info(
    "is_fiber_optic | fiber=%d (%.1f%%) churn=%.1f%% | outros=%d (%.1f%%) churn=%.1f%%",
    n_fiber,
    n_fiber / len(df) * 100,
    cr_fiber,
    n_outros,
    n_outros / len(df) * 100,
    cr_rest,
)

07:34:20 | INFO | is_fiber_optic | fiber=3090 (44.0%) churn=41.8% | outros=3931 (56.0%) churn=14.4%


### [4.7] Validações das features criadas

Suite de validações que garante a integridade da etapa de feature engineering:
1. Todas as 6 features foram criadas
2. Nenhuma tem valores nulos
3. Features binárias contêm apenas `{0, 1}`
4. `num_services` está dentro do range [0, 7]
5. Colunas originais ainda são strings — encoding **não** foi feito aqui


In [ ]:
NEW_FEATURES = [
    "num_services",
    "charges_per_month",
    "is_month_to_month",
    "tenure_group",
    "has_security_support",
    "is_fiber_optic",
]

BINARY_NEW_FEATURES = ["is_month_to_month", "has_security_support", "is_fiber_optic"]

# 1. Todas criadas
for feat in NEW_FEATURES:
    assert feat in df.columns, f"Feature ausente: '{feat}'"

# 2. Sem nulos nas features novas
nulos_novos = df[NEW_FEATURES].isnull().sum()
nulos_problema = nulos_novos[nulos_novos > 0]
assert len(nulos_problema) == 0, f"Nulos em features novas: {nulos_problema.to_dict()}"

# 3. Binárias contêm apenas 0 e 1
for feat in BINARY_NEW_FEATURES:
    valores_unicos = set(df[feat].unique())
    assert valores_unicos.issubset({0, 1}), (
        f"'{feat}' contém valores inesperados: {valores_unicos}"
    )

# 4. num_services no range correto
assert df["num_services"].between(0, len(SERVICES_COLS)).all(), (
    f"num_services fora do range [0, {len(SERVICES_COLS)}]"
)

# 5. Colunas originais preservadas como strings (encoding é do passo [5])
for col in ["online_security", "tech_support", "partner"]:
    assert df[col].dtype in ["object", "category"], (
        f"'{col}' foi encodada indevidamente — encoding é responsabilidade do passo [5]."
    )

features_novas = set(df.columns) - set(df_raw.columns) - {TARGET_COL}
logger.info(
    "VALIDAÇÕES FE OK | %d features criadas | strings originais preservadas | "
    "shape: %s → %s",
    len(NEW_FEATURES),
    shape_antes_fe,
    df.shape,
)

# Correlação das features novas com o target
logger.info("=== CORRELAÇÃO FEATURES NOVAS × TARGET ===")
numeric_new = [f for f in NEW_FEATURES if f != "tenure_group"]
for feat, corr in sorted(
    {f: round(df[f].corr(df[TARGET_COL]), 4) for f in numeric_new}.items(),
    key=lambda x: abs(x[1]),
    reverse=True,
):
    logger.info("  %s | corr_target=%+.4f", feat, corr)

07:34:20 | INFO | VALIDAÇÕES FE OK | 6 features criadas | strings originais preservadas | shape: (7021, 20) → (7021, 26)
07:34:20 | INFO | === CORRELAÇÃO FEATURES NOVAS × TARGET ===
07:34:20 | INFO |   charges_per_month | corr_target=+0.4094
07:34:20 | INFO |   is_month_to_month | corr_target=+0.4049
07:34:21 | INFO |   is_fiber_optic | corr_target=+0.3082
07:34:21 | INFO |   has_security_support | corr_target=-0.1817
07:34:21 | INFO |   num_services | corr_target=-0.0842


---

## [5] Preprocessing & Encoding

Etapa de transformação estatística: encoding de variáveis categóricas,
normalização e padronização das variáveis numéricas.

> **Princípio fundamental:** O preprocessor é ajustado (`fit`) **exclusivamente**
> no conjunto de treino e **apenas aplicado** (`transform`) no conjunto de teste.
> Qualquer vazamento de informação do teste para o treino invalidaria a avaliação.

---

### Por que não usar SMOTE+ENN?

O dataset tem **26.5% de churn** (~1.869 positivos em 7.043 amostras).
Este NÃO é um desbalanceamento severo que justifique oversampling artificial.

**Problema que o SMOTE causava:**
1. Balanceamento artificial para 51%/51% → `pos_weight ≈ 0.96 ≈ 1.0`
2. Com `pos_weight ≈ 1`, a `BCEWithLogitsLoss` trata FP e FN como igualmente graves
3. A assimetria real de custo (FN custa **58×** mais que FP) era ignorada pela loss
4. Thresholds ótimos caíam em 0.10–0.17 (irreais em produção)
5. MLP perdia sua vantagem estrutural

**Solução adotada:** Distribuição natural preservada (26.5%) +
`pos_weight = n_neg/n_pos ≈ 2.77` na `BCEWithLogitsLoss` da MLP.


### [5.1] Separação X / y

Separação das features do target antes do pipeline.


In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].astype(int)

logger.info(
    "Separação X/y | X.shape=%s | y.shape=%s | churn_rate=%.2f%%",
    X.shape,
    y.shape,
    y.mean() * 100,
)

07:34:21 | INFO | Separação X/y | X.shape=(7021, 25) | y.shape=(7021,) | churn_rate=26.45%


### [5.2] Definição dos grupos de transformação

O `ColumnTransformer` aplica transformações diferentes a grupos de colunas.
Esta é a **única célula do projeto onde o encoding é definido** — a configuração
é centralizada aqui e serializada como `preprocessor.pkl` para uso na API.

| Grupo | Transformação | Colunas | Justificativa |
|---|---|---|---|
| `num` | `log1p` → `Imputer` → `StandardScaler` | 5 numéricas | `total_charges` tem skewness=0.962; StandardScaler essencial para MLP |
| `bin` | `OrdinalEncoder(["No","Yes"])` | 12 binárias Yes/No | Garante No=0, Yes=1 independente da ordem |
| `ohe` | `OneHotEncoder(drop="first")` | 5 nominais | Evita dummy trap; seguro para categorias desconhecidas em produção |
| `pass` | `passthrough` | 3 features criadas | Já são int(0/1) — nenhuma transformação necessária |


In [ ]:
# ── Numéricas → log1p + Imputer(median) + StandardScaler ─────────────────────
COLS_NUM_PIPE = [
    "tenure_months",
    "monthly_charges",
    "total_charges",  # log1p aplicado internamente — skewness=0.962
    "num_services",
    "charges_per_month",
]

# ── Binárias Yes/No → OrdinalEncoder ──────────────────────────────────────────
# categorias fixas ["No","Yes"] → No=0, Yes=1 independente da ordem dos dados
# handle_unknown="use_encoded_value" + unknown_value=-1 → robusto em produção
COLS_BINARY_PIPE = [
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "paperless_billing",
    "senior_citizen",  # 0/1 original — OrdinalEncoder preserva semântica
]

# ── Nominais 3+ categorias → OneHotEncoder ────────────────────────────────────
# drop="first" evita multicolinearidade perfeita (dummy trap)
# handle_unknown="ignore" → não quebra se a API receber categoria desconhecida
COLS_OHE_PIPE = [
    "internet_service",  # DSL / Fiber optic / No (3 cats)
    "contract",  # Month-to-month / One year / Two year (3 cats)
    "payment_method",  # 4 métodos de pagamento
    "gender",  # Male / Female — mantido para análise de viés
    "tenure_group",  # novo / medio / longo — criada no passo [4]
]

# ── Features binárias criadas no passo [4] → passthrough ─────────────────────
# já são int(0/1) → nenhuma transformação necessária
COLS_PASS_PIPE = [
    "is_month_to_month",
    "has_security_support",
    "is_fiber_optic",
]

logger.info(
    "Grupos definidos | num=%d | binary=%d | ohe=%d | passthrough=%d | total=%d",
    len(COLS_NUM_PIPE),
    len(COLS_BINARY_PIPE),
    len(COLS_OHE_PIPE),
    len(COLS_PASS_PIPE),
    len(COLS_NUM_PIPE)
    + len(COLS_BINARY_PIPE)
    + len(COLS_OHE_PIPE)
    + len(COLS_PASS_PIPE),
)

07:34:21 | INFO | Grupos definidos | num=5 | binary=12 | ohe=5 | passthrough=3 | total=25


### [5.3] Validação de cobertura do ColumnTransformer

Verificamos que todas as colunas de `X` estão alocadas em exatamente um grupo.
Colunas sem grupo serão descartadas pelo `remainder="drop"` — o que é intencional,
mas deve ser monitorado para garantir que nenhuma feature relevante seja perdida acidentalmente.


In [ ]:
all_grouped = set(COLS_NUM_PIPE + COLS_BINARY_PIPE + COLS_OHE_PIPE + COLS_PASS_PIPE)
x_cols = set(X.columns)

sem_grupo = x_cols - all_grouped  # em X mas sem grupo → serão descartadas
grupo_sem_x = all_grouped - x_cols  # no grupo mas não em X → erro de config

if sem_grupo:
    logger.warning("Colunas em X sem grupo (serão descartadas): %s", sorted(sem_grupo))
if grupo_sem_x:
    logger.warning("Colunas em grupos mas ausentes em X: %s", sorted(grupo_sem_x))
if not sem_grupo and not grupo_sem_x:
    logger.info("Cobertura OK | todas as %d colunas de X cobertas.", len(x_cols))

07:34:21 | INFO | Cobertura OK | todas as 25 colunas de X cobertas.


### [5.4] Construção do ColumnTransformer (preprocessor)

Este é o objeto central de transformação do projeto. Uma vez serializado
em `models/preprocessor.pkl`, é carregado pela API FastAPI e pela MLP para
garantir **zero divergência** entre treino e inferência em produção.

**Fluxo do pipeline numérico:**
```
total_charges (raw) → log1p → valores mais simétricos
                    → StandardScaler → média≈0, std≈1
```

**`FunctionTransformer(feature_names_out="one-to-one")`:** Necessário para
que `get_feature_names_out()` funcione corretamente no sklearn ≥ 1.3.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    (
                        "log1p",
                        FunctionTransformer(
                            np.log1p,
                            validate=True,
                            feature_names_out="one-to-one",  # sklearn ≥ 1.3
                        ),
                    ),
                    ("scaler", StandardScaler()),
                ]
            ),
            COLS_NUM_PIPE,
        ),
        (
            "bin",
            OrdinalEncoder(
                categories=[["No", "Yes"]] * len(COLS_BINARY_PIPE),
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            COLS_BINARY_PIPE,
        ),
        (
            "ohe",
            OneHotEncoder(
                drop="first",
                sparse_output=False,
                handle_unknown="ignore",
            ),
            COLS_OHE_PIPE,
        ),
        (
            "pass",
            "passthrough",
            COLS_PASS_PIPE,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

logger.info("ColumnTransformer construído | 4 transformers: num, bin, ohe, pass")

07:34:21 | INFO | ColumnTransformer construído | 4 transformers: num, bin, ohe, pass


### [5.5] Train/Test Split estratificado

**Proporção:** 80% treino / 20% teste — padrão robusto para ~7k amostras.

**`stratify=y`:** Garante que a taxa de churn (~26.5%) seja preservada
em ambos os conjuntos. Sem estratificação, variações aleatórias poderiam
criar splits com distribuições de classe diferentes, distorcendo a avaliação.

**Verificação do delta:** Aceitamos no máximo 0.1pp de diferença entre
a taxa de churn no treino e no teste como garantia de estratificação adequada.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

delta_churn = abs(y_train.mean() - y_test.mean()) * 100
logger.info(
    "Split estratificado | "
    "train=%d (%.1f%%) churn=%.2f%% | "
    "test=%d  (%.1f%%) churn=%.2f%% | "
    "delta_churn=%.3fpp",
    len(X_train),
    len(X_train) / len(X) * 100,
    y_train.mean() * 100,
    len(X_test),
    len(X_test) / len(X) * 100,
    y_test.mean() * 100,
    delta_churn,
)
assert delta_churn < 0.5, (
    f"Estratificação inadequada: delta de churn={delta_churn:.3f}pp (esperado < 0.5pp)"
)
logger.info("Estratificação validada — delta < 0.5pp.")

07:34:21 | INFO | Split estratificado | train=5616 (80.0%) churn=26.44% | test=1405  (20.0%) churn=26.48% | delta_churn=0.035pp
07:34:21 | INFO | Estratificação validada — delta < 0.5pp.


### [5.6] Fit no treino → Transform em ambos

**Regra fundamental:** O `fit` acontece **apenas** no `X_train`.
O `X_test` recebe apenas `transform` com os parâmetros aprendidos no treino.

Isso simula o comportamento em produção: o modelo nunca "vê" os dados
de inferência durante o treinamento — nem para calcular médias, desvios ou
qualquer outra estatística de normalização.

Após a transformação, forçamos `float64` explicitamente pois o `ColumnTransformer`
com `passthrough` pode retornar `dtype=object` quando há mistura de tipos.


In [ ]:
# fit APENAS no treino — NUNCA no teste
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)

# força float64 — elimina resíduos de object dtype do passthrough
X_train_arr = X_train_arr.astype(np.float64)
X_test_arr = X_test_arr.astype(np.float64)

feature_names = preprocessor.get_feature_names_out()

logger.info(
    "Transformação concluída | X_train=%s | X_test=%s | n_features=%d | dtype=%s",
    X_train_arr.shape,
    X_test_arr.shape,
    len(feature_names),
    X_train_arr.dtype,
)
logger.info("Features finais (%d): %s", len(feature_names), list(feature_names))

07:34:21 | INFO | Transformação concluída | X_train=(5616, 30) | X_test=(1405, 30) | n_features=30 | dtype=float64
07:34:21 | INFO | Features finais (30): ['tenure_months', 'monthly_charges', 'total_charges', 'num_services', 'charges_per_month', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'paperless_billing', 'senior_citizen', 'internet_service_Fiber optic', 'internet_service_No', 'contract_One year', 'contract_Two year', 'payment_method_Credit card (automatic)', 'payment_method_Electronic check', 'payment_method_Mailed check', 'gender_Male', 'tenure_group_medio', 'tenure_group_novo', 'is_month_to_month', 'has_security_support', 'is_fiber_optic']


### [5.7] Validações pós-transformação

Suite de contratos que garante a qualidade do dataset transformado antes
de qualquer persistência. Falhas aqui indicam problemas no pipeline
de encoding que invalidariam o treinamento dos modelos.


In [ ]:
# Converte para DataFrame com nomes de features para facilitar inspeção
X_train_df = (
    pd.DataFrame(X_train_arr, columns=feature_names)
    .astype(float)
    .reset_index(drop=True)
)
X_test_df = (
    pd.DataFrame(X_test_arr, columns=feature_names).astype(float).reset_index(drop=True)
)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# ── Contratos obrigatórios ─────────────────────────────────────────────────
assert X_train_df.isnull().sum().sum() == 0, "Nulos em X_train após transformação"
assert X_test_df.isnull().sum().sum() == 0, "Nulos em X_test após transformação"
assert X_train_df.shape[1] == X_test_df.shape[1], (
    f"n_features diferente: train={X_train_df.shape[1]} test={X_test_df.shape[1]}"
)
assert X_train_df.select_dtypes(include="object").empty, (
    f"Colunas object em X_train: {list(X_train_df.select_dtypes('object').columns)}"
)

# ── Verificação de colunas suspeitas (object residual) ─────────────────────
cols_object = [col for col in X_train_df.columns if X_train_df[col].dtype == object]
if cols_object:
    logger.warning("Colunas object residuais (inesperado): %s", cols_object)
else:
    logger.info("dtype OK — nenhuma coluna object residual.")

logger.info(
    "Validações pós-transformação OK | "
    "X_train=%s | X_test=%s | n_features=%d | sem nulos | sem object",
    X_train_df.shape,
    X_test_df.shape,
    len(feature_names),
)

X_train_df.describe().round(3)

07:34:21 | INFO | dtype OK — nenhuma coluna object residual.
07:34:21 | INFO | Validações pós-transformação OK | X_train=(5616, 30) | X_test=(1405, 30) | n_features=30 | sem nulos | sem object


,tenure_months,monthly_charges,total_charges,num_services,charges_per_month,partner,dependents,phone_service,multiple_lines,online_security,...,contract_Two year,payment_method_Credit card (automatic),payment_method_Electronic check,payment_method_Mailed check,gender_Male,tenure_group_medio,tenure_group_novo,is_month_to_month,has_security_support,is_fiber_optic
count,5616.000,5616.000,5616.000,5616.000,5616.000,5616.000,5616.000,5616.000,5616.000,5616.000,...,5616.000,5616.000,5616.000,5616.000,5616.0,5616.000,5616.000,5616.000,5616.000,5616.000
mean,0.000,-0.000,-0.000,0.000,0.000,0.487,0.231,0.906,0.331,0.287,...,0.241,0.216,0.339,0.226,0.5,0.375,0.308,0.548,0.420,0.440
std,1.000,1.000,1.000,1.000,1.000,0.500,0.422,0.292,0.641,0.452,...,0.428,0.412,0.473,0.418,0.5,0.484,0.462,0.498,0.494,0.496
min,-2.651,-1.877,-2.593,-2.584,-1.343,0.000,0.000,0.000,-1.000,0.000,...,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,0.000
25%,-0.646,-0.756,-0.610,-1.161,-0.693,0.000,0.000,1.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.0,0.000,0.000,0.000,0.000,0.000
50%,0.311,0.385,0.188,0.262,-0.342,0.000,0.000,1.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.0,0.000,0.000,1.000,0.000,0.000
75%,0.870,0.799,0.842,0.720,0.555,1.000,0.000,1.000,1.000,1.000,...,0.000,0.000,1.000,0.000,1.0,1.000,1.000,1.000,1.000,1.000
max,1.086,1.275,1.381,1.684,3.259,1.000,1.000,1.000,1.000,1.000,...,1.000,1.000,1.000,1.000,1.0,1.000,1.000,1.000,1.000,1.000


---

## [6] Persistência dos Datasets Processados

Geração dos 4 arquivos de saída em `data/processed/`:

| Arquivo | Formato | Conteúdo | Uso |
|---|---|---|---|
| `train.parquet` | Parquet | X_train + target (float64) | Baselines sklearn, análise exploratória |
| `test.parquet`  | Parquet | X_test + target (float64)  | Avaliação final dos modelos |
| `train.npy`     | NumPy   | array float32 com target na última coluna | MLP PyTorch |
| `test.npy`      | NumPy   | array float32 com target na última coluna | MLP PyTorch |

**Por que dois formatos?**
- **Parquet:** Preserva nomes de colunas, dtypes e metadados — ideal para
  pipelines sklearn e análise exploratória
- **NumPy float32:** Mais eficiente para PyTorch (`torch.from_numpy` → `FloatTensor`)
  sem conversão de dtype em tempo de treinamento


### [6.1] Parquet — treino e teste

O target é adicionado como última coluna em ambos os arquivos.
Validação pós-save relê o arquivo e compara shape para garantir
integridade da escrita.


In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# ── train.parquet ─────────────────────────────────────────────────────────────
train_df = X_train_df.copy()
train_df[TARGET_COL] = y_train.values
train_path = DATA_PROCESSED / "train.parquet"
train_df.to_parquet(train_path, index=False)

train_check = pd.read_parquet(train_path)
assert train_check.shape == train_df.shape, (
    f"Erro na persistência do train.parquet | esperado={train_df.shape} lido={train_check.shape}"
)
logger.info(
    "train.parquet salvo e validado | path=%s | shape=%s | churn=%.2f%%",
    train_path,
    train_df.shape,
    train_df[TARGET_COL].mean() * 100,
)

# ── test.parquet ──────────────────────────────────────────────────────────────
test_df = X_test_df.copy()
test_df[TARGET_COL] = y_test.values
test_path = DATA_PROCESSED / "test.parquet"
test_df.to_parquet(test_path, index=False)

test_check = pd.read_parquet(test_path)
assert test_check.shape == test_df.shape, (
    f"Erro na persistência do test.parquet | esperado={test_df.shape} lido={test_check.shape}"
)
logger.info(
    "test.parquet  salvo e validado | path=%s | shape=%s | churn=%.2f%%",
    test_path,
    test_df.shape,
    test_df[TARGET_COL].mean() * 100,
)

07:34:21 | INFO | train.parquet salvo e validado | path=C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\tech_challenge\data\processed\train.parquet | shape=(5616, 31) | churn=26.44%
07:34:21 | INFO | test.parquet  salvo e validado | path=C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\tech_challenge\data\processed\test.parquet | shape=(1405, 31) | churn=26.48%


### [6.2] NumPy `.npy` — treino e teste (float32)

Arrays NumPy com o target concatenado como **última coluna**.
`float32` é o dtype nativo do PyTorch (`torch.FloatTensor`) —
evita conversão implícita durante o treinamento e reduz uso de memória.

**Convenção de leitura:**
```python
data = np.load("train.npy")
X    = data[:, :-1]   # todas as colunas exceto a última
y    = data[:, -1]    # última coluna = target
```


In [ ]:
# ── train.npy ─────────────────────────────────────────────────────────────────
train_npy = np.column_stack(
    [
        X_train_arr.astype(np.float32),
        y_train.values.astype(np.float32),
    ]
)
train_npy_path = DATA_PROCESSED / "train.npy"
np.save(train_npy_path, train_npy)

train_npy_check = np.load(train_npy_path)
assert train_npy_check.shape == train_npy.shape, (
    f"Erro na persistência do train.npy | esperado={train_npy.shape} lido={train_npy_check.shape}"
)
assert train_npy_check.dtype == np.float32, (
    f"dtype incorreto: esperado float32, lido {train_npy_check.dtype}"
)
logger.info(
    "train.npy salvo e validado | shape=%s | dtype=%s | target_col=%d",
    train_npy.shape,
    train_npy.dtype,
    train_npy.shape[1] - 1,
)

# ── test.npy ──────────────────────────────────────────────────────────────────
test_npy = np.column_stack(
    [
        X_test_arr.astype(np.float32),
        y_test.values.astype(np.float32),
    ]
)
test_npy_path = DATA_PROCESSED / "test.npy"
np.save(test_npy_path, test_npy)

test_npy_check = np.load(test_npy_path)
assert test_npy_check.shape == test_npy.shape, (
    f"Erro na persistência do test.npy | esperado={test_npy.shape} lido={test_npy_check.shape}"
)
logger.info(
    "test.npy  salvo e validado | shape=%s | dtype=%s | target_col=%d",
    test_npy.shape,
    test_npy.dtype,
    test_npy.shape[1] - 1,
)

07:34:21 | INFO | train.npy salvo e validado | shape=(5616, 31) | dtype=float32 | target_col=30
07:34:21 | INFO | test.npy  salvo e validado | shape=(1405, 31) | dtype=float32 | target_col=30


### [6.3] Serialização do `preprocessor.pkl`

O `preprocessor` (ColumnTransformer ajustado) é serializado com `joblib`
em `models/preprocessor.pkl`. Este é o **objeto carregado pela API FastAPI**
na Etapa 3 para transformar dados de entrada antes de enviá-los ao modelo.

**Validação de round-trip:** Recarregamos o objeto salvo e verificamos
que produz outputs numericamente idênticos ao original. Divergência aqui
indicaria corrupção na serialização.


In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
preprocessor_path = MODELS_DIR / "preprocessor.pkl"

joblib.dump(preprocessor, preprocessor_path)

# Validação de round-trip — confirma reprodutibilidade da serialização
preprocessor_reloaded = joblib.load(preprocessor_path)
sample_input = X_test.iloc[:10]
out_original = preprocessor.transform(sample_input)
out_reloaded = preprocessor_reloaded.transform(sample_input)

assert np.allclose(out_original, out_reloaded, atol=1e-10), (
    "Preprocessor recarregado produz resultados diferentes — serialização corrompida."
)

logger.info(
    "preprocessor.pkl serializado e validado | path=%s | round-trip OK",
    preprocessor_path,
)

07:34:21 | INFO | preprocessor.pkl serializado e validado | path=C:\Users\victo\Desktop\MLOPS\AULAS\tech_challenge_fase_1\tech_challenge\models\preprocessor.pkl | round-trip OK


### [6.4] Resumo final

Consolidação de todas as métricas relevantes do pipeline para registro
e auditoria. Inclui metadados que serão logados no MLflow nos notebooks
de modelagem.


In [ ]:
# ── pos_weight para BCEWithLogitsLoss da MLP ──────────────────────────────────
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
pos_weight = round(n_neg / n_pos, 4)

logger.info("=" * 60)
logger.info("RESUMO FINAL DO PIPELINE")
logger.info("=" * 60)
logger.info("Dataset original    | shape=%s", df_raw.shape)
logger.info("Dataset pós-limpeza | shape=%s", df.shape)
logger.info("Features finais     | n=%d", len(feature_names))
logger.info("Features criadas    | %s", NEW_FEATURES)
logger.info("-" * 60)
logger.info(
    "Train split | n=%d (%.1f%%) | churn=%.2f%% | n_pos=%d | n_neg=%d",
    len(y_train),
    len(y_train) / len(y) * 100,
    y_train.mean() * 100,
    n_pos,
    n_neg,
)
logger.info(
    "Test  split | n=%d (%.1f%%) | churn=%.2f%%",
    len(y_test),
    len(y_test) / len(y) * 100,
    y_test.mean() * 100,
)
logger.info("-" * 60)
logger.info(
    "pos_weight MLP      | %.4f  (n_neg/n_pos = %d/%d)", pos_weight, n_neg, n_pos
)
logger.info("Arquivos gerados    | train.parquet, test.parquet, train.npy, test.npy")
logger.info("Preprocessor        | models/preprocessor.pkl")
logger.info("SMOTE+ENN           | REMOVIDO (26.5%% churn não requer oversampling)")
logger.info("=" * 60)
logger.info("2 preprocessing_completo concluído com sucesso.")

# ── Exibe feature names para referência (via logging) ────────────────────────
logger.info("--- Features finais (%d) ---", len(feature_names))
for i, name in enumerate(feature_names):
    logger.info("  [%02d] %s", i, name)
logger.info("--- Arquivos gerados ---")
for path_out in [
    train_path,
    test_path,
    train_npy_path,
    test_npy_path,
    preprocessor_path,
]:
    logger.info("  %s", path_out)

07:34:21 | INFO | ============================================================
07:34:21 | INFO | RESUMO FINAL DO PIPELINE
07:34:21 | INFO | ============================================================
07:34:21 | INFO | Dataset original    | shape=(7043, 20)
07:34:21 | INFO | Dataset pós-limpeza | shape=(7021, 26)
07:34:21 | INFO | Features finais     | n=30
07:34:21 | INFO | Features criadas    | ['num_services', 'charges_per_month', 'is_month_to_month', 'tenure_group', 'has_security_support', 'is_fiber_optic']
07:34:21 | INFO | ------------------------------------------------------------
07:34:21 | INFO | Train split | n=5616 (80.0%) | churn=26.44% | n_pos=1485 | n_neg=4131
07:34:21 | INFO | Test  split | n=1405 (20.0%) | churn=26.48%
07:34:21 | INFO | ------------------------------------------------------------
07:34:21 | INFO | pos_weight MLP      | 2.7818  (n_neg/n_pos = 4131/1485)
07:34:21 | INFO | Arquivos gerados    | train.parquet, test.parquet, train.npy, test.npy
07:34:21 | I